### ⚙️ Initial Setup

In [0]:
import pyspark.sql.functions as sf
import matplotlib.pyplot as plt
import seaborn as sns

# style like R ggplot
plt.style.use("ggplot")

In [0]:
# base volume path
BASE_DIR = "/Volumes/workspace/default/home-credit-default-risk"

In [0]:
# reading data
prev_app = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/previous_application.csv")
)

In [0]:
print(f"({prev_app.count()}, {len(prev_app.columns)})")

In [0]:
# schema and dtype
prev_app.printSchema()

### 1. 📫 Previous Application

The **`previous_application`** table is a record of every past loan request a client submitted to Home Credit before applying for their current loan. Each row represents one individual loan request, capturing key details such as the type of product requested (like a cash loan, credit card, or store installment for appliances), how much money the borrower asked for versus what the bank actually approved, and whether the application was ultimately accepted, refused, or canceled.

#### SK_ID_PREV (Previous Loan / Application ID):

Role: Primary Key of the `previous_application` table.

Description: A unique identifier for a specific historical application or contract submitted to Home Credit in the past. Every single past loan attempt gets its own distinct SK_ID_PREV.

#### SK_ID_CURR (Current Applicant ID):

Role: Foreign Key linking back to application_train / application_test.

Description: A unique identifier for the individual customer/person applying for the current loan.

In [0]:
prev_app.show(5)

#### Top Applicants with the Highest Number of Historical Applications.

In [0]:
top_applicants = (
    prev_app.groupBy("SK_ID_CURR")
    .agg(sf.count("SK_ID_PREV").alias("PREV_APP_COUNT"))
    .orderBy(sf.col("PREV_APP_COUNT").desc())
    .limit(10)
)

top_applicants_pd = top_applicants.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=top_applicants_pd,
    x="SK_ID_CURR",
    y="PREV_APP_COUNT",
    ax=ax,
    edgecolor="black",
    palette="plasma",
    hue="SK_ID_CURR",
)

# Add count labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

ax.set_title(
    "Top 10 Applicants by Number of Previous Applications",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

ax.set_xlabel("Applicant ID")
ax.set_ylabel("Number of Previous Applications")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---

In [0]:
prev_app.groupBy("NAME_CONTRACT_TYPE").count().show()

#### What is NAME_CONTRACT_TYPE?
It indicates the specific product category requested in that historical application (SK_ID_PREV).

##### Categories Breakdown  
* **Consumer loans (POS):** Financing taken out directly at a store or checkout counter to buy a specific physical product (e.g., a phone, appliance, or furniture).
* **Cash loans:** Direct cash transferred straight into the borrower's bank account to spend on whatever they need.
* **Revolving loans:** Open credit lines (like credit cards) where the borrower can repeatedly draw funds, pay them back, and borrow again up to a set limit.
* **Unused offer / XNA:** Pre-approved loan offers or system-generated options that were canceled, expired, or turned down by the applicant before any money was disbursed.

In [0]:
contract_df = (
    prev_app
    .groupBy("NAME_CONTRACT_TYPE")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(7, 7))

ax.pie(
    contract_df["count"],
    labels=contract_df["NAME_CONTRACT_TYPE"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.set_title(
    "Distribution of Previous Application Contract Types",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

---

`AMT_ANNUITY` in the `previous_application` table represents the **monthly installment amount** for that specific past loan application.

* **What it shows:** The exact amount the applicant was scheduled (or expected) to pay each month for that historical loan (`SK_ID_PREV`).
* **Difference from Main Table:** In `application_train/test`, `AMT_ANNUITY` is the monthly payment for the *new* requested loan. Here, it is the monthly payment for a *past* loan attempt.
* **Why Missing Values Occur:** It is often null for applications that were **refused**, **canceled**, or **unused**, since no active loan agreement or payment schedule was generated.

In [0]:
# summary stats
prev_app.select("AMT_ANNUITY").describe().show()

In [0]:
# Boxplot
prev_app.select("AMT_ANNUITY").plot.box()

In [0]:
quantiles = prev_app.approxQuantile(
    "AMT_ANNUITY",
    [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0],
    0.01
)

print(f"0%: {quantiles[0]:,.2f}")
print(f"25%: {quantiles[1]:,.2f}")
print(f"50%: {quantiles[2]:,.2f}")
print(f"75%: {quantiles[3]:,.2f}")
print(f"90%: {quantiles[4]:,.2f}")
print(f"95%: {quantiles[5]:,.2f}")
print(f"90%: {quantiles[6]:,.2f}")
print(f"100%: {quantiles[7]:,.2f}")

In [0]:
# Histogram Plot
prev_app.select("AMT_ANNUITY").plot.hist(bins=10, title="Histogram Plot of AMT_ANNUITY")

#### Kurtosis and Skewness

In [0]:
prev_app.select(
    sf.skewness("AMT_ANNUITY").alias("skewness"),
    sf.kurtosis("AMT_ANNUITY").alias("kurtosis")
).show()

##### 1. Skewness = 2.69 (Strong Right Skew)
The vast majority of past loan payments are clustered at lower values, but a long right tail of very large monthly payments pulls the distribution to the right.

Intuition: Most historical applicants took out small consumer loans with small monthly payments, while a few applicants took large loans with very high monthly commitments.

##### 2. Kurtosis = 15.07 (Heavy-Tailed / Leptokurtic)
Any kurtosis value significantly above 3.0 indicates extreme tail behavior and sharp peaking compared to a normal distribution.

Intuition: The dataset has severe, high-value outliers rather than a smooth, bell-shaped spread.

**Leptokurtic** describes a distribution with a **sharper, taller central peak** and **"fat tails"** (extreme outliers) compared to a standard normal bell curve.

* **The Tall Peak:** Most of our data points are tightly squished together around a low/average value (e.g., most past loan annuities were small).
* **The Fat Tails:** Outliers happen way more often than normal probability predicts. Instead of extreme values fading away quickly, we get a long trail of unexpectedly huge numbers.

---

#### Let's Explore AMT_APPLICATION and AMT_CREDIT
`AMT_APPLICATION` represents the exact loan amount the borrower originally requested in their past application, while `AMT_CREDIT` is the final credit amount actually approved and granted by Home Credit for that same contract. In short, **`AMT_APPLICATION` is what the customer asked for, and `AMT_CREDIT` is what the bank gave them.**

The relationship between these two columns provides a direct window into Home Credit's past risk decisions. When `AMT_CREDIT` is lower than `AMT_APPLICATION`, it means the risk team down-sized or "haircut" the loan request to limit exposure. When the two values match, the applicant was deemed fully creditworthy for their requested amount.

In [0]:
prev_app.select("AMT_APPLICATION", "AMT_CREDIT").show(5)

#### Requested vs. Approved Amount Analysis (`AMT_APPLICATION` vs `AMT_CREDIT`)

In [0]:
credit_vs_app_df = prev_app.withColumn(
    "CREDIT_VS_APP",
    sf.when(sf.col("AMT_APPLICATION") > sf.col("AMT_CREDIT"), "Haircut (Requested > Approved)")
     .when(sf.col("AMT_APPLICATION") < sf.col("AMT_CREDIT"), "Upsold (Requested < Approved)")
     .when(sf.col("AMT_APPLICATION") == sf.col("AMT_CREDIT"), "Exact Match")
     .otherwise("Null / Zero")
).groupBy("CREDIT_VS_APP").agg(
    sf.count("*").alias("count"),
    sf.round(sf.count("*") / prev_app.count() * 100, 2).alias("percentage")
)
credit_vs_app_df.show(truncate=-1)

In [0]:
# convert Spark DataFrame to Pandas
credit_vs_app_pd = credit_vs_app_df.toPandas()

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=credit_vs_app_pd,
    x="CREDIT_VS_APP",
    y="percentage",
    ax=ax,
    edgecolor="black",
    palette="pastel",
    hue="CREDIT_VS_APP",
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3
    )

ax.set_title(
    "Requested vs Approved Credit",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Credit Comparison")
ax.set_ylabel("Percentage (%)")

plt.xticks(rotation=15)

plt.tight_layout()
plt.show()


#### Business Intuition: Requested vs. Approved Loan Amounts

* **`AMT_APPLICATION > AMT_CREDIT` (Downsized / Haircut)**
* **Scenario:** The borrower requested $10,000, but Home Credit approved only $7,000.
* **Business Signal:** The underwriting process flagged potential risk—such as high existing debt or tight monthly cash flow—and capped loan exposure. A pattern of frequent haircuts suggests an applicant who routinely requests more credit than their financial profile supports.

* **`AMT_APPLICATION == AMT_CREDIT` (Exact Approval)**
* **Scenario:** The applicant requested $10,000 and received $10,000.
* **Business Signal:** A clean approval where the customer's income, credit score, and financial history fully supported the requested obligation without requiring risk adjustments.

* **`AMT_APPLICATION < AMT_CREDIT` (Financed Add-Ons or Upsell)**
* **Scenario:** The approved credit amount ended up higher than the initial request.
* **Business Signal:** This typically occurs when ancillary products—such as loan protection insurance or administrative fees—are capitalized directly into the total credit balance. In other cases, highly qualified borrowers are offered a larger pre-approved limit than what they initially asked for.

---

#### AMT_DOWN_PAYMENT  
It is basically the downpayment submitted by applicant during loan processing. It is highly related with `NAME_CONTRACT_TYPE` let's see how!

In [0]:
prev_app.groupBy("NAME_CONTRACT_TYPE").agg(
    sf.count("*").alias("total_rows"),
    sf.count("AMT_DOWN_PAYMENT").alias("non_null_count"),
    sf.sum(sf.when(sf.col("AMT_DOWN_PAYMENT").isNull(), 1).otherwise(0)).alias("null_count"),
    sf.round(
        (sf.sum(sf.when(sf.col("AMT_DOWN_PAYMENT").isNull(), 1).otherwise(0)) / sf.count("*")) * 100,
        2
    ).alias("null_pct")
).show(5)

#### Structural Missingness Analysis: `AMT_DOWN_PAYMENT`

* **Key Observation:** Over 94% of Cash Loans and 99% of Revolving Loans have a `null` value for `AMT_DOWN_PAYMENT`.
* **Root Cause:** This missingness is **Structural (Missing by Design)**. Cash loans and credit card lines do not involve retail items at the point of sale, so an upfront cash down payment is legally and product-wise non-applicable.
* **Semantic Meaning:** 
  * For **Cash / Revolving Loans**: `Null` means **Not Applicable** (no product down payment mechanism exists).
  * For **Consumer Loans (POS)**: `Null` means either a **0% down payment promotion** or a contract that was **refused/canceled** before cash collection.

---

#### WEEKDAY_APPR_PROCESS_START

It records the exact day of the week from Monday through Sunday on which a customer submitted a past loan application. Because this timestamp is generated automatically by the system whenever an application is initiated, the column is complete with no missing values.

In [0]:
prev_app.groupBy("WEEKDAY_APPR_PROCESS_START") \
    .agg(
        sf.count("*").alias("app_count"),
        sf.round((sf.count("*") / prev_app.count()) * 100, 2).alias("percentage")
    ) \
    .orderBy(sf.col("app_count").desc()) \
    .show()

There a dip in Sunday because it is a standard day off (non-working day) for regular bank branches, back-office operations, and corporate centers in the regions where Home Credit operates (primarily Central/Eastern Europe and emerging markets).

#### What is FLAG_LAST_APPL_PER_CONTRACT?

Multiple application records can occasionally be created for a single loan contract due to clerk re-keying errors, customer re-submissions, or system crashes. `FLAG_LAST_APPL_PER_CONTRACT` marks whether a record is the final submission for that specific loan contract.
* Y (Yes - Final Application)
* N (No - Duplicate / Superseded)

An entry marked `N` means another record exists for that same contract marked `Y` representing the final version.

In [0]:
prev_app.groupBy("FLAG_LAST_APPL_PER_CONTRACT").count().show()

In [0]:
flag_df = (
    prev_app
    .groupBy("FLAG_LAST_APPL_PER_CONTRACT")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(7, 7))

wedges, _, autotexts = ax.pie(
    flag_df["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "width": 0.4,
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    flag_df["FLAG_LAST_APPL_PER_CONTRACT"],
    title="Last Application",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "Last Application per Contract",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

*Note: I will probably drop these features during data cleaning*

---

#### NAME_CONTRACT_STATUS 
It indicates the final disposition or outcome of a previous loan application submitted by the customer.

##### Value Breakdown & Business Meaning
* `Approved`: Home Credit accepted the application and issued the loan.

* `Refused`: Home Credit rejected the application due to credit risk, policy constraints, or failed verification checks. A history of multiple refused applications is a strong risk indicator for default.

* `Canceled`: The applicant voluntarily withdrew the application before final agreement execution or decisioning.

* `Unused offer`: Home Credit approved or extended a credit offer, but the customer opted not to take or activate it.

In [0]:
prev_app.groupBy("NAME_CONTRACT_STATUS").count().show()

In [0]:
status_df = (
    prev_app
    .groupBy("NAME_CONTRACT_STATUS")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=status_df,
    x="NAME_CONTRACT_STATUS",
    y="count",
    ax=ax,
    edgecolor="black",
    palette="Set2",
    hue="NAME_CONTRACT_STATUS",
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=3
    )

ax.set_title(
    "Distribution of Contract Status",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Contract Status")
ax.set_ylabel("Number of Applications")

plt.tight_layout()
plt.show()

#### Contact Type on Contract Status

In [0]:
cross_df = (
    prev_app
    .crosstab("NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS")
)

status_cols = [
    c for c in cross_df.columns
    if c != "NAME_CONTRACT_TYPE_NAME_CONTRACT_STATUS"
]

cross_pct = cross_df.withColumn(
    "total",
    sum(sf.col(c) for c in status_cols)
)

for c in status_cols:
    cross_pct = cross_pct.withColumn(
        f"{c}_pct",
        sf.round(sf.col(c) / sf.col("total") * 100, 2)
    )

cross_pct.show(truncate=False)

#### Conclusion
##### Consumer Loans (POS) have the highest approval rate (85.92%)
Fact: 626,470 out of 729,151 consumer loan applications were approved, with virtually zero cancellations (0.21%).  
Business Reason: POS loans finance specific retail products (like electronics or furniture), providing lower risk for the bank.

##### Cash Loans have a high cancellation rate (35.93%) and the lowest approval rate (41.81%)
Fact: Over 268,000 cash loan applications were canceled before completion, and over 165,000 were refused.  
Business Reason: Cash loan applicants frequently shop around, change their minds, or drop out when asked for income documentation.

---

#### NAME_PAYMENT_TYPE 
It records the repayment method chosen by the client to pay the monthly installments for their previous loan application.

* `Cash`: The applicant pays installments in physical cash over the counter at a bank branch, post office, or retail payment point.

* `Non-cash`: Automatic direct debit or electronic bank transfer set up from the applicant's personal bank account.

* `Cashless`: Repayment method where loan installments are automatically deducted directly from an employee's salary by their employer and transferred electronically to the lender before the net paycheck is issued.

* `XNA (Not Applicable / Unspecified)`: Applications that were refused, canceled, or unfulfilled before repayment terms were finalized, or where the payment option was left unrecorded.

In [0]:
prev_app.groupBy("NAME_PAYMENT_TYPE").count().show(truncate=-1)

In [0]:
payment_df = (
    prev_app
    .groupBy("NAME_PAYMENT_TYPE")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 8))

wedges, texts, autotexts = ax.pie(
    payment_df["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    payment_df["NAME_PAYMENT_TYPE"],
    title="Payment Type",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title(
    "Distribution of Payment Types",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

---

#### NAME_YIELD_GROUP 
It categorizes the interest rat charged by Home Credit on a past loan into standardized pricing tiers. Because raw interest rates are anonymized in this dataset, this column serves as a clean proxy for the historical interest rate bucket applied to an application.

1. `low_action`: Special promotional or discounted low interest rate offered during campaigns.

2. `low_normal`: Standard low interest rate tier assigned to low-risk clients or specific competitive products.

3. `middle`: Standard/average interest rate tier.

4. `high`: High interest rate tier charged to higher-risk applicants or subprime loans to offset default risk.

5. `XNA`: Unassigned or not applicable (primarily occurs on refused, canceled, or unfulfilled applications where no rate was finalized).

In [0]:
prev_app.groupBy("NAME_YIELD_GROUP").count().show()

In [0]:
yield_df = (
    prev_app
    .groupBy("NAME_YIELD_GROUP")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=yield_df,
    x="NAME_YIELD_GROUP",
    y="count",
    ax=ax,
    edgecolor="black",
    palette="Set3",
    hue="NAME_YIELD_GROUP",

)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=3
    )

ax.set_title(
    "Distribution of Yield Groups",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Yield Group")
ax.set_ylabel("Number of Applications")

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---

#### DAYS_FIRST_DRAWING Means
It is the date when the customer actually got the money (or when the store got paid) for a past loan.

##### Interpretation
All dates in this dataset count backward from today (the current application):

* 0 = Today (the day of the current application).

* -30 = Money was disbursed 30 days ago.

* -365 = Money was disbursed 1 year ago.

`Example`: If DAYS_FIRST_DRAWING is -100, it means the payout for that past loan happened 100 days ago.

If a past loan was refused, canceled, or never activated, money was never paid out.

Instead of leaving the column empty or writing 0 (which would mean "paid out today"), the database puts a placeholder number: `365243` (approx. 1,000 years in the future).

`365243` = "Money was NEVER paid out" (loan was rejected, canceled, or unused).

In [0]:
prev_app.filter(sf.col("DAYS_FIRST_DRAWING") == 365243).count()

In [0]:
prev_app.groupBy("DAYS_FIRST_DRAWING").count().orderBy("count", ascending=False).limit(5).show()